# 2. Regression ve Hyperparameter Optimization

Yazarlar: Furkan Cantürk ve Gemini 3 Pro  
İlk sürüm: 21.11.2025

Bir beton karışımının bileşenleri ve yaşı biliniyor. Basınç dayanımını tahmin edebilir miyiz? Önce basit modeller kuracağız. Sonra XGBoost’ta ağaçların karmaşıklığını sınırlayıp yeni karışımlardaki tahmin hatasının nasıl değiştiğini ölçeceğiz.

**Öğrenme kazanımları**

- Bir regression probleminde feature ve target sütunlarını, train, validation ve test rollerini ayırt etmek.
- RMSE ile bir baseline kurmak ve eğitim ile validation hatasını birlikte okuyarak overfitting belirtisini tanımak.
- XGBoost'ta ağaçların karmaşıklığını iki hyperparameter üzerinden açıklamak ve `GridSearchCV` ile bir ayar seçmek.
- HPO öncesi ve sonrası performansı aynı örneklerde karşılaştırmak ve model seçiminin etkisini bağımsız bir değerlendirmeyle ölçmek.
- Eğitilmiş modelin hangi girdilere dayandığını permutation importance ile incelemek.

**İçindekiler**

| Bölüm | Konu |
|---|---|
| 1–2 | Ders ortamı ve beton dayanımı regression problemi |
| 3–4 | Veri ayrımı, keşif ve preprocessing |
| 5–6 | RMSE, cross-validation ve baseline deneyi |
| 7 | Hyperparameter araması ve görseller |
| 8 | HPO etkisinin ölçümü |
| 9–10 | Final test, feature importance, alıştırma ve MLflow |

**Ön bilgi:** Temel Python ve pandas. **Önerilen çalışma biçimi:** Önce soruyu yanıtlamaya çalışın, sonra kod hücresini çalıştırıp çıktıyı yorumlayın.

## 1. Ders ortamı

Kurulum için [dersin README dosyasına](../README.md#environment-setup) bakın. Kullanacağımız CSV depo içinde bulunur. Defteri yukarıdan aşağıya çalıştırın. Plotly grafiklerinde değerleri okumak için fareyi noktaların üzerine getirebilirsiniz.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

COLOR_TRAIN = "#2563a5"
COLOR_VALIDATION = "#c36a18"
COLOR_BEFORE = "#64748b"
COLOR_AFTER = "#007f73"

## 2. Problem ve veri

### 2.1 Bir satır neyi temsil ediyor?

Bir satır, belirli bir beton karışımının belirli bir yaştaki dayanım ölçümüdür. Örneğin `Cement` ve `Water` bir metreküp karışımdaki miktarları (kg/m³), `Age` ise yaşı gün olarak gösterir. Hedef `Concrete compressive strength`, megapaskal (MPa) cinsinden basınç dayanımıdır.

| Girdi | Ölçüm |
|---|---|
| `Cement`, `Water` ve diğer beş bileşen | Karışım miktarı (kg/m³) |
| `Age` | Beton yaşı (gün) |
| `Concrete compressive strength` | Hedef dayanım (MPa) |

**Kaynak:** I-Cheng Yeh (1998), [Concrete Compressive Strength, UCI Machine Learning Repository](https://doi.org/10.24432/C5PK67). [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Depodaki CSV, [UCI'nin yayımladığı CSV](https://archive.ics.uci.edu/static/public/165/data.csv) ile aynıdır.

In [2]:
data_path = Path("data/concrete_compressive_strength.csv")
if not data_path.is_file():
    data_path = Path("exercises") / data_path

data = pd.read_csv(data_path)
data.sample(10)

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Concrete compressive strength
376,376.0,0.0,0.0,214.6,0.0,1003.5,762.4,28,31.97
504,491.0,26.0,123.0,201.0,3.9,822.0,699.0,3,29.55
195,194.7,0.0,100.5,165.6,7.5,1006.4,905.9,14,24.99
716,116.0,173.0,0.0,192.0,0.0,909.8,891.9,7,10.09
542,255.0,0.0,0.0,192.0,0.0,889.8,945.0,90,21.86
972,145.7,172.6,0.0,181.9,3.4,985.8,816.8,28,23.74
937,313.3,0.0,113.0,178.5,8.0,1001.9,688.7,28,36.80
706,153.0,102.0,0.0,192.0,0.0,888.0,943.1,90,26.32
680,102.0,153.0,0.0,192.0,0.0,887.0,942.0,28,17.28
685,133.0,200.0,0.0,192.0,0.0,927.4,839.2,7,13.66


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1030 entries, 0 to 1029
Data columns (total 9 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Cement                         1030 non-null   float64
 1   Blast Furnace Slag             1030 non-null   float64
 2   Fly Ash                        1030 non-null   float64
 3   Water                          1030 non-null   float64
 4   Superplasticizer               1030 non-null   float64
 5   Coarse Aggregate               1030 non-null   float64
 6   Fine Aggregate                 1030 non-null   float64
 7   Age                            1030 non-null   int64  
 8   Concrete compressive strength  1030 non-null   float64
dtypes: float64(8), int64(1)
memory usage: 72.6 KB


In [4]:
duplicate_count = data.duplicated().sum()
data = data.drop_duplicates().reset_index(drop=True)
print(f"Tam tekrar olarak çıkarılan satır: {duplicate_count}")
print(f"Ölçüm sayısı: {len(data)} | Sütun sayısı: {data.shape[1]}")

Tam tekrar olarak çıkarılan satır: 25
Ölçüm sayısı: 1005 | Sütun sayısı: 9


### 2.2 Feature ve target ayrımı

`X` modelin göreceği yedi karışım bileşenini ve `Age` sütununu tutar. `y` ölçülen dayanımı tutar. Gerçek dayanım gelecekte tahmin etmek istediğimiz değer olduğu için yalnızca `y` içinde yer alır.

**Düşünün.** Bir beton karışımının 7. ve 28. gündeki dayanımı ölçülmüşse bu iki satırda hangi girdiler aynı kalır?

<details><summary>Yanıtı aç</summary>Karışım bileşenleri aynı kalır. Age ve ölçülen dayanım değişebilir.</details>

In [5]:
target_column = "Concrete compressive strength"
X = data.drop(columns=target_column)
y = data[target_column]
print("X boyutu:", X.shape)
print("y boyutu:", y.shape)

X boyutu: (1005, 8)
y boyutu: (1005,)


## 3. Train ve test ayrımı

Amacımız **yeni bir karışımın** dayanımını tahmin etmek. Satırları rastgele ayırsaydık aynı karışımın 7 günlük ölçümü eğitimde, 28 günlük ölçümü testte kalabilirdi. Bu yüzden yedi bileşeni aynı olan satırlara aynı grup numarasını veriyoruz.

`GroupShuffleSplit` grupların yaklaşık %20'sini test için ayırır. Her grubun tüm ölçümleri aynı veri kümesinde kalır. Grupların büyüklükleri farklı olabileceği için testteki satır oranı %20'den farklı olabilir. Eğitim kümesini baseline, HPO ve model seçimi için kullanırız.

In [6]:
ingredients = X.columns.drop("Age").tolist()
recipe_groups = X.groupby(ingredients, sort=False).ngroup()
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=recipe_groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
train_groups = recipe_groups.iloc[train_idx]

print(f"Train: {len(X_train)} ölçüm | Test: {len(X_test)} ölçüm")
print(f"Train: {train_groups.nunique()} farklı karışım | Test: {recipe_groups.iloc[test_idx].nunique()} farklı karışım")
display(X_train[["Age", "Cement", "Water"]].assign(**{target_column: y_train}).head())

Train: 800 ölçüm | Test: 205 ölçüm
Train: 341 farklı karışım | Test: 86 farklı karışım


,Age,Cement,Water,Concrete compressive strength
1,28,540.0,162.0,61.89
2,270,332.5,228.0,40.27
3,365,332.5,228.0,41.05
4,360,198.6,192.0,44.30
5,90,266.0,228.0,47.03


## 4. Keşif ve preprocessing

### 4.1 Beton yaşı ve dayanımı

Solda eğitim kümesindeki tüm ölçümler, sağda en fazla farklı yaşta ölçülen karışımın dayanımı var. Sağdaki grafik, bileşenler sabitken dayanımın yaşla nasıl değiştiğini gösterir. Aynı yaşta birden fazla ölçüm varsa noktada ortalama dayanım gösterilir. Çizgi sabit bir hızla mı yükseliyor?

In [7]:
from plotly.subplots import make_subplots

exploration = pd.DataFrame({"Age": X_train["Age"], "Strength": y_train, "Recipe": train_groups})
age_counts = exploration.groupby("Recipe")["Age"].nunique()
example_recipe = exploration.loc[exploration["Recipe"] == age_counts.idxmax()]
example_recipe = example_recipe.groupby("Age", as_index=False)["Strength"].mean()

fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=["Bütün training ölçümleri", "Bir karışımın yaşa göre dayanımı"])
fig.add_scatter(x=exploration["Age"], y=exploration["Strength"], mode="markers",
                marker={"color": COLOR_TRAIN, "opacity": 0.35}, row=1, col=1)
fig.add_scatter(x=example_recipe["Age"], y=example_recipe["Strength"], mode="lines+markers",
                line={"color": COLOR_AFTER}, row=1, col=2)
fig.update_xaxes(title_text="Age (gün)")
fig.update_yaxes(title_text="Basınç dayanımı (MPa)", row=1, col=1)
fig.update_yaxes(title_text="Basınç dayanımı (MPa)", showticklabels=True, row=1, col=2)
fig.update_layout(showlegend=False)
fig.show()

### 4.2 Hangi dönüşümlere ihtiyaç var?

Sütun türlerini ve eksik değer sayısını kontrol edelim. Kontrol tablosunda sütunların sayısal olduğunu ve eksik değer bulunmadığını göreceksiniz. Bu nedenle `OneHotEncoder` ya da `SimpleImputer` gerekmiyor. `LinearRegression` örneğinde `StandardScaler` ile bir `Pipeline` kuracağız. Decision Tree sıralı eşiklerle böldüğü için ölçekleme gerektirmez.

**Pipeline kuralı:** Her fold için scaler'ın ortalaması ve standart sapması o fold'un training parçasında öğrenilir. Validation parçasında aynı dönüşüm uygulanır. Böylece validation verisinin istatistikleri training aşamasına sızmaz.

In [8]:
data_check = pd.DataFrame({
    "Data type": X_train.dtypes,
    "Missing values": X_train.isna().sum(),
})
display(data_check)
linear_pipeline = make_pipeline(StandardScaler(), LinearRegression())

,Data type,Missing values
Cement,float64,0
Blast Furnace Slag,float64,0
Fly Ash,float64,0
Water,float64,0
Superplasticizer,float64,0
Coarse Aggregate,float64,0
Fine Aggregate,float64,0
Age,int64,0


## 5. Performansı nasıl ölçeceğiz?

### 5.1 RMSE'yi elle hesaplayalım

İki gerçek dayanım ölçümü 100 ve 200, tahminler 110 ve 180 olsun. Hatalar +10 ve −20. Karelerini alıp ortaladığımızda 250, karekökünü aldığımızda **15,8 MPa** elde ederiz. RMSE büyüdükçe hata artar. Kare alma büyük hataların etkisini artırır.

In [9]:
example_y = [100, 200]
example_predictions = [110, 180]
print(f"RMSE = {root_mean_squared_error(example_y, example_predictions):.1f} MPa")

RMSE = 15.8 MPa


### 5.2 Üç fold ile cross-validation

`GroupKFold`, eğitim verisini A, B ve C olarak üç parçaya ayırır. Bir turda A'da ölçüp B ve C'de öğreniriz. Sonraki turlarda validation görevi B'ye ve C'ye geçer. Aynı karışımın tüm ölçümleri tek parçada kalır.

Aynı üç fold'u bütün modeller için saklıyoruz. Böylece model tahminlerini aynı validation verisinde karşılaştırıyoruz.

In [10]:
cv = GroupKFold(n_splits=3, shuffle=True, random_state=123)
outer_folds = list(cv.split(X_train, y_train, groups=train_groups))
scoring = "neg_root_mean_squared_error"

fold_roles = [[int(fold == part) for part in range(3)] for fold in range(3)]
fig = px.imshow(
    fold_roles, x=["A", "B", "C"], y=["Tur 1", "Tur 2", "Tur 3"],
    aspect="auto", color_continuous_scale=[COLOR_TRAIN, COLOR_VALIDATION],
    title="Her parça bir kez validation görevi görür",
    labels={"x": "Training kümesindeki parçalar", "y": ""},
)
fig.update_traces(text=np.where(np.array(fold_roles) == 1, "Validation", "Train"),
                  texttemplate="%{text}", textfont={"color": "white"}, hoverinfo="skip")
fig.update_layout(coloraxis_showscale=False, height=290)
fig.show()

## 6. Baseline deneyi ve overfitting

### 6.1 Model ayarları

`DummyRegressor` training ölçümlerinin ortalamasını herkese verir. Bu, modellerden beklediğimiz en temel karşılaştırma noktası. `LinearRegression` ağırlıklı bir toplam öğrenir. `DecisionTreeRegressor` ise örnekleri ardışık eşiklerle böler. XGBoost, LightGBM ve CatBoost birden çok ağacı birleştiren ek karşılaştırmalardır.

Decision Tree başlangıçta `max_depth=None` ve `min_samples_leaf=1` kullanır. XGBoost için de özellikle derin ağaçlara izin veren bir başlangıç ayarı seçtik (`max_depth=12`, `min_child_weight=1`). Böylece 7. bölümde karmaşıklığı sınırlamanın etkisini ölçebileceğiz. **Tahmin edin.** En düşük training RMSE hangi modelde olur? Validation sıralaması da aynı olur mu?

In [11]:
baseline_models = {
    "Dummy Regressor": DummyRegressor(),
    "Linear Regression": linear_pipeline,
    "Decision Tree": DecisionTreeRegressor(max_depth=None, min_samples_leaf=1, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, max_depth=12, min_child_weight=1,
                            learning_rate=0.1, random_state=42, n_jobs=1),
    "LightGBM": LGBMRegressor(n_estimators=100, random_state=42, verbosity=-1, n_jobs=1),
    "CatBoost": CatBoostRegressor(iterations=100, random_seed=42, verbose=False,
                                  thread_count=1, allow_writing_files=False),
}

### 6.2 HPO öncesindeki modelleri çalıştır

`cross_validate` her modelin training ve validation skorlarını fold bazında döndürür. Scikit-learn yüksek skoru tercih ettiği için `neg_root_mean_squared_error` ters işaretli değer üretir. Tabloya koyarken işareti çeviriyoruz. Birazdan bu `baseline` tablosundaki değerleri HPO sonrası sonuçlarla eşleştireceğiz.

In [12]:
baseline_records = []
for name, model in baseline_models.items():
    cv_results = cross_validate(
        model, X_train, y_train, cv=outer_folds, scoring=scoring,
        return_train_score=True, return_estimator=(name == "Decision Tree"),
        n_jobs=1, error_score="raise",
    )
    if name == "Decision Tree":
        first_fold_tree = cv_results["estimator"][0]
    for fold_number, (train, validation) in enumerate(
        zip(-cv_results["train_score"], -cv_results["test_score"]), start=1
    ):
        baseline_records.append({"Model": name, "Fold": fold_number,
                                    "Train RMSE": train, "Validation RMSE": validation})

baseline_fold = pd.DataFrame(baseline_records)
baseline = baseline_fold.groupby("Model", sort=False)[["Train RMSE", "Validation RMSE"]].mean()
baseline["RMSE gap"] = baseline["Validation RMSE"] - baseline["Train RMSE"]
display(baseline.round(2))

,Train RMSE,Validation RMSE,RMSE gap
Model,,,
Dummy Regressor,15.83,15.88,0.05
Linear Regression,10.03,10.60,0.58
Decision Tree,1.01,8.54,7.53
XGBoost,1.01,6.49,5.48
LightGBM,2.37,6.03,3.66
CatBoost,2.02,5.68,3.66


In [13]:
baseline_plot = baseline.reset_index().melt(
    id_vars="Model", value_vars=["Train RMSE", "Validation RMSE"],
    var_name="Metric", value_name="RMSE",
)
fig = px.bar(
    baseline_plot, x="Model", y="RMSE", color="Metric", barmode="group",
    color_discrete_map={"Train RMSE": COLOR_TRAIN, "Validation RMSE": COLOR_VALIDATION},
    title="Before HPO: Train ve validation hataları", labels={"RMSE": "RMSE (MPa)"},
)
fig.show()

### 6.3 Decision Tree'nin Train RMSE'si nasıl bu kadar düşer?

**Train RMSE**, ağacın fit sırasında gördüğü örneklerdeki tahmin hatasıdır. Her örnek ağacın öğrendiği eşiklerden geçip bir **leaf** düğümüne ulaşır. Varsayılan `squared_error` ölçütüyle ağacın o leaf için tahmini, oraya düşen training target değerlerinin ortalamasıdır.

| Leaf içindeki gerçek target değerleri | Leaf'in tahmini | Bu örneklerdeki hatalar |
|---|---:|---|
| 100 ve 200 | 150 | +50 ve −50 |
| Yalnızca 100 | 100 | 0 |
| Yalnızca 200 | 200 | 0 |

`max_depth=None` derinliğe açık bir üst sınır koymaz, `min_samples_leaf=1` ise tek örnekli leaf düğümlerine izin verir. Ağaç ilk satırdaki iki örneği ayrı leaf düğümlerine gönderebilirse her biri için kendi target değerini tahmin eder. Bu durum bütün training örneklerinde gerçekleşirse **her tahmin hatası sıfır, dolayısıyla RMSE de sıfır** olur. Farklı target değerlerine sahip iki örneğin feature değerleri aynıysa ağaç onları ayıramaz. Tek örnekli olmayan bir leaf de içindeki target değerleri aynıysa sıfır hata verebilir.

Aşağıdaki hücre 6.2'de eğitilen ağacın **ilk fold'daki training parçasını** inceler. `apply` her örneğin ulaştığı leaf numarasını, `predict` ise o leaf'in tahminini verir.

In [14]:
first_train_idx, _ = outer_folds[0]
X_first_train = X_train.iloc[first_train_idx]
y_first_train = y_train.iloc[first_train_idx]

leaf_inspection = pd.DataFrame({
    "Leaf ID": first_fold_tree.apply(X_first_train),
    "Actual target": y_first_train.to_numpy(),
    "Prediction": first_fold_tree.predict(X_first_train),
})
leaf_inspection["Error"] = leaf_inspection["Prediction"] - leaf_inspection["Actual target"]
leaf_sizes = leaf_inspection["Leaf ID"].value_counts()
print(f"Training örneği: {len(leaf_inspection)} | Tek örnekli leaf: {(leaf_sizes == 1).sum()}")
print(f"Hatası sıfır olan training örneği: {(leaf_inspection['Error'] == 0).sum()}")
display(leaf_inspection.head())

if (leaf_sizes > 1).any():
    shared_leaf = leaf_sizes.idxmax()
    print("Aynı leaf'e düşen örnekler")
    display(leaf_inspection.loc[leaf_inspection["Leaf ID"] == shared_leaf])

Training örneği: 507 | Tek örnekli leaf: 469
Hatası sıfır olan training örneği: 495


,Leaf ID,Actual target,Prediction,Error
0,933,61.89,61.89,0.0
1,563,40.27,40.27,0.0
2,564,41.05,41.05,0.0
3,750,44.30,44.30,0.0
4,591,47.03,47.03,0.0


Aynı leaf'e düşen örnekler


,Leaf ID,Actual target,Prediction,Error
197,961,61.07,57.486667,-3.583333
198,961,56.14,57.486667,1.346667
199,961,55.25,57.486667,2.236667


**Düşünün.** Bileşenleri ve yaşı aynı olan iki ölçümün gerçek dayanımları 40 ve 44 MPa ise ağaç bunları ayrı yapraklara gönderebilir mi? Sadece bu iki ölçüm bir yaprağa düşerse tahmin nedir?

<details><summary>Yanıtı aç</summary>

Aynı feature değerleri aynı yolu izler. Bu yaprakta tahmin 42 MPa, iki örneğin hataları +2 ve −2 MPa olur. Gerçek deneyde bunun gibi örnekler olduğu için training RMSE sıfırın üzerinde kalabilir. Aşırı uyumu görmek için validation RMSE'yi de okuyun.

</details>

## 7. HPO ve hyperparameter diyagramları

### 7.1 XGBoost'ta bir ağacı ne sınırlar?

XGBoost, hatayı azaltmak için ağaçları sırayla ekler. Her ağacın çok ayrıntılı kurallar öğrenmesine izin verirsek model eğitim verisine fazla uyabilir. İki ayarla bunu kontrollü biçimde değiştireceğiz.

| Hyperparameter | Nasıl okunur? |
|---|---|
| `max_depth` | Küçük değer, bir karar yolundaki bölme sayısını sınırlar. |
| `min_child_weight` | Bir bölmenin oluşturacağı çocuk düğümdeki asgari Hessian toplamıdır. Bu ağırlıksız, squared error deneyinde daha büyük değer, her çocukta daha fazla örnek gerektirir. |

Örneğin `max_depth=2` bir yol üzerinde en fazla iki bölmeye izin verir. `min_child_weight=5` bu deneyde yalnızca iki training örneği içeren bir çocuk oluşturmayı engeller. XGBoost'ta `n_estimators` ve `learning_rate` sabit. Böylece 6 derinlik × 4 `min_child_weight` = **24 aday** boyunca iki ayarın etkisini izleyebiliriz.

In [15]:
depths = [2, 3, 4, 6, 8, 12]
child_weights = [1, 3, 5, 10]
parameter_grids = {
    "Decision Tree": {"max_depth": [2, 3, 4, 6, 10, None], "min_samples_leaf": [1, 5, 10, 20, 40]},
    "XGBoost": {"max_depth": depths, "min_child_weight": child_weights},
    "LightGBM": {"n_estimators": [100, 250], "num_leaves": [7, 15, 31], "learning_rate": [0.03, 0.1]},
    "CatBoost": {"iterations": [100, 300, 600], "depth": [4, 6], "learning_rate": [0.03, 0.1]},
}

### 7.2 Adayları karşılaştır ve ayarı seç

7. bölümde bütün training kümesi üzerinden yaptığımız arama, 9. bölümde teste götürülecek modeli seçmek içindir. `best_params_` seçilen ayarları, `cv_results_` her adayın ölçümlerini verir. Dummy Regressor ve Linear Regression için bu deneyde HPO uygulamıyoruz.

In [16]:
searches = {}
for name, grid in parameter_grids.items():
    search = GridSearchCV(
        baseline_models[name], grid, cv=outer_folds, scoring=scoring,
        return_train_score=True, n_jobs=1, error_score="raise",
    )
    search.fit(X_train, y_train)
    searches[name] = search
    print(f"{name}: {search.best_params_}")

Decision Tree: {'max_depth': 10, 'min_samples_leaf': 1}
XGBoost: {'max_depth': 6, 'min_child_weight': 3}
LightGBM: {'learning_rate': 0.1, 'n_estimators': 250, 'num_leaves': 15}
CatBoost: {'depth': 4, 'iterations': 600, 'learning_rate': 0.1}


### 7.3 Complexity curve

`min_child_weight=1` sabitken `max_depth` arttığında training ve validation hatalarını birlikte izleyin. Validation eğrisi yükselmeye başladığı halde training eğrisi düşüyorsa daha derin ağaçlar eğitim verisine daha iyi uyuyor ve validation verisinde daha çok hata yapıyor.

In [17]:
xgb_search = searches["XGBoost"]
candidates = pd.DataFrame(xgb_search.cv_results_)
candidates["max_depth"] = candidates["param_max_depth"].astype(int)
candidates["min_child_weight"] = candidates["param_min_child_weight"].astype(int)
candidates["Train RMSE"] = -candidates["mean_train_score"]
candidates["Validation RMSE"] = -candidates["mean_test_score"]

curve = candidates.loc[candidates["min_child_weight"] == 1].melt(
    id_vars="max_depth", value_vars=["Train RMSE", "Validation RMSE"],
    var_name="Metric", value_name="RMSE",
)
fig = px.line(
    curve, x="max_depth", y="RMSE", color="Metric", markers=True,
    color_discrete_map={"Train RMSE": COLOR_TRAIN, "Validation RMSE": COLOR_VALIDATION},
    title="XGBoost complexity curve (min_child_weight=1)",
    labels={"RMSE": "RMSE (MPa)"},
)
fig.update_xaxes(tickvals=depths)
fig.show()

### 7.4 Heatmap

Her hücre bir `max_depth` ve `min_child_weight` birleşiminin validation RMSE'sidir. Bir satırda ilerleyerek `min_child_weight` etkisini, bir sütunda ilerleyerek `max_depth` etkisini okuyun. Açık sarı düşük hatayı gösterir. Beyaz çerçeve baseline'ı, siyah çerçeve GridSearchCV'nin seçimini işaretler.

In [18]:
heatmap_data = candidates.pivot(index="max_depth", columns="min_child_weight", values="Validation RMSE")
heatmap_data = heatmap_data.reindex(index=depths, columns=child_weights)
fig = px.imshow(
    heatmap_data, x=[str(value) for value in child_weights], y=[str(value) for value in depths], text_auto=".1f",
    aspect="auto", color_continuous_scale="Viridis_r",
    labels={"x": "min_child_weight", "y": "max_depth", "color": "Validation RMSE"},
    title="XGBoost: arama alanındaki 24 aday",
)
best_params = xgb_search.best_params_
fig.add_scatter(x=["1"], y=["12"], name="Baseline", mode="markers",
                marker={"symbol": "square-open", "color": "white", "size": 39, "line_width": 2})
fig.add_scatter(x=[str(best_params["min_child_weight"])], y=[str(best_params["max_depth"])],
                name="Seçilen aday", mode="markers",
                marker={"symbol": "square-open", "color": "black", "size": 39, "line_width": 2})
fig.update_xaxes(type="category")
fig.update_yaxes(type="category")
fig.update_layout(
    legend=dict(orientation="h", x=0.5, xanchor="center", y=0.02, yanchor="bottom", yref="container"),
    margin=dict(b=100),
)
fig.show()

### 7.5 Parallel Coordinates

Her çizgi aynı 24 adaydan biridir. Soldan sağa `max_depth`, `min_child_weight`, Train RMSE ve Validation RMSE'yi takip edin. Son eksende düşük hataları işaretlemek için eksenin üzerinde bir aralık seçin. Seçili adaylar ilk iki eksende hangi ayarlarda toplanıyor?

In [19]:
fig = go.Figure(go.Parcoords(
    line={"color": candidates["Validation RMSE"], "colorscale": "Viridis_r", "showscale": True,
          "colorbar": {"title": "Validation<br>RMSE"}},
    dimensions=[
        {"label": "max_depth", "values": candidates["max_depth"],
         "tickvals": depths},
        {"label": "min_child_weight", "values": candidates["min_child_weight"],
         "tickvals": child_weights},
        {"label": "Train RMSE", "values": candidates["Train RMSE"]},
        {"label": "Validation RMSE", "values": candidates["Validation RMSE"]},
    ],
))
fig.update_layout(title="XGBoost: hyperparameter ve hata profilleri", height=440,
                  margin={"l": 75, "r": 105, "t": 90, "b": 55})
fig.show()

### 7.6 HPO sürecini bağımsız fold'larda ölçmeye hazırlan

Bir ayarı seçmek için kullandığımız validation verisi, o seçimin performansını olduğundan iyi gösterebilir. HPO sürecini değerlendirmek için **nested cross-validation** kullanıyoruz. Aşağıda 6. bölümdeki üç fold'dan birini düşünün:

| Adım | Veri | İşlem |
|---|---|---|
| 1 | Dış training parçası | Karışım gruplarını koruyarak üç iç fold oluştur, aday ayarları karşılaştır. |
| 2 | Dış training parçasının tamamı | İç fold'larda seçilen XGBoost modelini yeniden fit et. |
| 3 | Dış validation parçası | Fit edilen ağacın RMSE'sini ölç. |

Bu döngü üç kez tekrarlanır. Baseline ölçümleri 6. bölümde aynı dış fold'larda yapılmıştı. Dolayısıyla 8. bölümde HPO öncesi ve sonrası tahminleri aynı validation verisi üzerinde karşılaştırılabilir. Ayrı final test kümesi henüz açılmadı.

In [20]:
inner_cv = GroupKFold(n_splits=3, shuffle=True, random_state=42)
tuned_folds = []

for fold_number, (outer_train_idx, outer_val_idx) in enumerate(outer_folds, start=1):
    X_outer_train, y_outer_train = X_train.iloc[outer_train_idx], y_train.iloc[outer_train_idx]
    X_outer_val, y_outer_val = X_train.iloc[outer_val_idx], y_train.iloc[outer_val_idx]
    inner_search = GridSearchCV(
        baseline_models["XGBoost"], parameter_grids["XGBoost"],
        cv=inner_cv, scoring=scoring, n_jobs=1, error_score="raise",
    ).fit(X_outer_train, y_outer_train, groups=train_groups.iloc[outer_train_idx])
    tuned_folds.append({
        "Fold": fold_number,
        "Train RMSE": root_mean_squared_error(y_outer_train, inner_search.predict(X_outer_train)),
        "Validation RMSE": root_mean_squared_error(y_outer_val, inner_search.predict(X_outer_val)),
        "Best parameters": inner_search.best_params_,
    })

## 8. HPO öncesi ve sonrası

### 8.1 XGBoost için bağımsız karşılaştırma

Önce şu iki soruyu yanıtlayın. HPO sonrası **dış validation RMSE** düştü mü? **Validation − Train RMSE farkı** küçüldü mü? İlk ölçü yeni örneklerdeki tahmin hatasını, ikincisi modelin training verisinde ne kadar daha iyi göründüğünü gösterir. Bir modeli gereğinden fazla basitleştirmek farkı küçültürken validation hatasını artırabilir, bu nedenle iki ölçüyü birlikte okuyun.

In [21]:
xgb_before = baseline_fold.loc[baseline_fold["Model"] == "XGBoost"].drop(columns="Model")
xgb_after = pd.DataFrame(tuned_folds)
paired = xgb_before.merge(xgb_after, on="Fold", suffixes=(" before", " after"), validate="one_to_one")
paired["Validation RMSE change"] = paired["Validation RMSE after"] - paired["Validation RMSE before"]
display(paired.round(2))

metrics = ["Train RMSE", "Validation RMSE"]
xgb_summary = pd.DataFrame({
    "Before HPO": xgb_before[metrics].mean(),
    "After HPO": xgb_after[metrics].mean(),
}).T.rename_axis("Stage")
xgb_summary["RMSE gap"] = xgb_summary["Validation RMSE"] - xgb_summary["Train RMSE"]
display(xgb_summary.round(2))

before, after = xgb_summary["Validation RMSE"]
print(f"Ortalama dış validation RMSE: {before:.2f} → {after:.2f} MPa")
print(f"Validation RMSE değişimi: {after - before:+.2f} MPa ({100 * (after - before) / before:+.1f}%)")
print(f"İyileşen fold sayısı: {(paired['Validation RMSE change'] < 0).sum()}/{len(paired)}")

,Fold,Train RMSE before,Validation RMSE before,Train RMSE after,Validation RMSE after,Best parameters,Validation RMSE change
0,1,0.77,7.69,2.40,6.89,"{'max_depth': 4, 'min_child_weight': 1}",-0.80
1,2,1.23,5.74,2.77,4.98,"{'max_depth': 4, 'min_child_weight': 3}",-0.76
2,3,1.03,6.04,2.67,5.69,"{'max_depth': 4, 'min_child_weight': 5}",-0.35


,Train RMSE,Validation RMSE,RMSE gap
Stage,,,
Before HPO,1.01,6.49,5.48
After HPO,2.61,5.85,3.24


Ortalama dış validation RMSE: 6.49 → 5.85 MPa
Validation RMSE değişimi: -0.64 MPa (-9.8%)
İyileşen fold sayısı: 3/3


In [22]:
paired_plot = paired.melt(
    id_vars="Fold", value_vars=["Validation RMSE before", "Validation RMSE after"],
    var_name="Stage", value_name="Validation RMSE",
)
paired_plot["Fold"] = "Fold " + paired_plot["Fold"].astype(str)
fig = px.bar(
    paired_plot, x="Fold", y="Validation RMSE", color="Stage", barmode="group",
    color_discrete_map={"Validation RMSE before": COLOR_BEFORE, "Validation RMSE after": COLOR_AFTER},
    title="XGBoost: aynı dış validation fold'larında önce ve sonra",
    labels={"Validation RMSE": "Validation RMSE (MPa)"},
)
fig.show()

**Beklenen okuma.** Örnek çalıştırmada dış validation RMSE yaklaşık **6,49 → 5,85 MPa** düştü. Derinliği 12 olan baseline'ın yerine iç aramalar üç fold'da da `max_depth=4` seçti. Train RMSE biraz artarken üç dış fold'un da validation RMSE'si azaldı. Bu tablo daha sınırlı XGBoost modelinin dış validation verisinde daha iyi tahmin yaptığını gösteriyor.

### 8.2 Diğer modeller ve final seçimi

Bu tabloda 6. bölümdeki baseline puanlarını ve 7.2'deki arama puanlarını yan yana görüyoruz. HPO sütunu ayarların seçildiği fold'lardan gelir ve seçim nedeniyle iyimser olabilir. Bu tabloyu **model seçimi** için kullanıyoruz. XGBoost üzerinde HPO etkisine ilişkin bağımsız ölçüm 8.1'deki dış fold sonuçlarıdır.

In [23]:
results = []
for name, baseline_row in baseline.iterrows():
    search = searches.get(name)
    before = baseline_row["Validation RMSE"]
    after = -search.best_score_ if search is not None else before
    results.append({
        "Model": name, "Baseline RMSE": before, "Tuned RMSE": after if search is not None else np.nan,
        "RMSE change": after - before if search is not None else np.nan,
        "Selection RMSE": after, "Best parameters": search.best_params_ if search is not None else {},
    })

model_results = pd.DataFrame(results).sort_values("Selection RMSE").reset_index(drop=True)
display(model_results.round(2))

,Model,Baseline RMSE,Tuned RMSE,RMSE change,Selection RMSE,Best parameters
0,CatBoost,5.68,5.29,-0.39,5.29,"{'depth': 4, 'iterations': 600, 'learning_rate..."
1,XGBoost,6.49,5.74,-0.75,5.74,"{'max_depth': 6, 'min_child_weight': 3}"
2,LightGBM,6.03,5.82,-0.21,5.82,"{'learning_rate': 0.1, 'n_estimators': 250, 'n..."
3,Decision Tree,8.54,8.46,-0.09,8.46,"{'max_depth': 10, 'min_samples_leaf': 1}"
4,Linear Regression,10.60,NaN,NaN,10.60,{}
5,Dummy Regressor,15.88,NaN,NaN,15.88,{}


## 9. Final test

### 9.1 Test performansı

8.2'de en düşük `Selection RMSE` alan modeli eğitim kümesinin tamamında eğitip teste götürüyoruz. Sonucu aynı test kümesinde Linear Regression ile karşılaştıracağız. Test puanlarını kullanarak ayar veya model seçimi yapmayacağız.

RMSE, hatayı MPa cinsinden özetler. R², testteki dayanım değerlerinin değişkenliğini ortalama tahminine göre modelin ne kadar açıkladığını gösterir. Örneğin R² = 0,8 olduğunda kareli hata toplamı, test ortalamasını her satıra tahmin etmenin kareli hata toplamından %80 daha düşüktür.

In [24]:
selected_name = model_results.loc[0, "Model"]
if selected_name in searches:
    selected_model = searches[selected_name].best_estimator_
else:
    selected_model = baseline_models[selected_name].fit(X_train, y_train)

predictions = selected_model.predict(X_test)
test_rmse = root_mean_squared_error(y_test, predictions)
test_r2 = r2_score(y_test, predictions)
linear_predictions = linear_pipeline.fit(X_train, y_train).predict(X_test)
test_results = pd.DataFrame([
    {"Model": "Linear Regression", "Test RMSE (MPa)": root_mean_squared_error(y_test, linear_predictions),
     "Test R²": r2_score(y_test, linear_predictions)},
    {"Model": selected_name, "Test RMSE (MPa)": test_rmse, "Test R²": test_r2},
]).drop_duplicates(subset="Model")
print(f"Validation ile seçilen model: {selected_name}")
display(test_results.round(3))

Validation ile seçilen model: CatBoost


,Model,Test RMSE (MPa),Test R²
0,Linear Regression,10.724,0.637
1,CatBoost,5.905,0.890


**Grafiği okuyun:** Kesikli çizgi `tahmin = gerçek` çizgisidir. Çizginin üzerindeki noktalarda model yüksek, altındakilerde düşük tahmin yapmıştır. Dikey uzaklık tahmin hatasının büyüklüğüdür. Yüksek dayanım değerlerinde hangi yönde hata daha sık görülüyor?

In [25]:
prediction_data = pd.DataFrame({"Actual strength (MPa)": y_test, "Prediction (MPa)": predictions})
prediction_data["Error"] = prediction_data["Prediction (MPa)"] - prediction_data["Actual strength (MPa)"]
lower = min(y_test.min(), predictions.min())
upper = max(y_test.max(), predictions.max())
fig = px.scatter(prediction_data, x="Actual strength (MPa)", y="Prediction (MPa)", hover_data=["Error"],
                 title=f"Final test: {selected_name}")
fig.add_scatter(x=[lower, upper], y=[lower, upper], mode="lines", name="tahmin = gerçek",
                line={"color": COLOR_BEFORE, "dash": "dash"}, hoverinfo="skip")
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.show()

### 9.2 Feature importance

Seçilen model dayanımı tahmin ederken en çok hangi girdilerden yararlanıyor? **Permutation importance** için test kümesinde her seferinde bir sütunun değerlerini satırlar arasında karıştırırız. Diğer sütunlar ve gerçek dayanım değerleri yerinde kalır. Aynı modelle tekrar tahmin yapıp RMSE'nin ilk değere göre kaç MPa değiştiğini hesaplarız. Bunu her sütun için 20 kez yineleriz.

Çubuklar RMSE'deki ortalama değişimi, hata çizgileri ise tekrarlar arasındaki standart sapmayı gösterir. Büyük pozitif değer, modelin o girdiden çok yararlandığını gösterir. Sıfıra yakın değer, hatanın benzer kaldığını gösterir. Negatif değer, karıştırma sonrasında ortalama hatanın düştüğü anlamına gelir. Birbiriyle ilişkili girdiler benzer bilgi taşıyorsa tek tek önemleri düşük görünebilir. Bu grafik seçilmiş modelin tahminlerini yorumlamak içindir.

**Düşünün.** En yüksek çubuk hangi girdiye ait? Sütunu karıştırmak RMSE'yi ortalama kaç MPa artırıyor?

In [26]:
from sklearn.inspection import permutation_importance

importance = permutation_importance(
    selected_model, X_test, y_test, scoring=scoring, n_repeats=20, random_state=42,
)
feature_importance = pd.DataFrame({
    "Girdi": X_test.columns,
    "Ortalama RMSE değişimi (MPa)": importance.importances_mean,
    "Standart sapma (MPa)": importance.importances_std,
}).sort_values("Ortalama RMSE değişimi (MPa)")

fig = px.bar(
    feature_importance, x="Ortalama RMSE değişimi (MPa)", y="Girdi",
    error_x="Standart sapma (MPa)", orientation="h",
    title=f"Permutation importance: {selected_name}",
    color_discrete_sequence=[COLOR_AFTER],
)
fig.add_vline(x=0, line_dash="dot", line_color=COLOR_BEFORE)
fig.show()

## 10. Uygulama ve deney kaydı

1. **Baseline grafiği.** Decision Tree ve XGBoost için train ve validation RMSE arasındaki farkı bulun. Hangi model validation verisinde daha iyi tahmin yapıyor?
2. **Heatmap.** `max_depth=12` satırında `min_child_weight=1` ile `5` arasındaki validation RMSE değişimini okuyun. Parallel Coordinates üzerinde bu adayların training hatalarını da bulun.
3. **Nested cross-validation.** Dış fold tablosundan XGBoost ayarlarının en çok işe yaradığı fold'u bulun. Seçilen ayarları baseline ile karşılaştırın.
4. **Final test.** Seçilen model ile Linear Regression arasındaki RMSE farkını MPa cinsinden hesaplayın. Model bu iki test skoruna bakılmadan hangi ölçüte göre seçilmişti?

<details><summary>Kontrol yanıtlarını aç</summary>

1. Validation RMSE sütununda XGBoost'un hata değeri daha düşük.
2. Heatmap validation hatasını, Parallel Coordinates ise her iki hatayı gösterir. İki ayarı karşılaştırırken `max_depth` sabit kalır.
3. `Validation RMSE change` en negatif olan satırı bulun. Ayarları `max_depth=12`, `min_child_weight=1` ile karşılaştırın.
4. İki test RMSE değeri arasındaki farkı alın. Model seçimi 8.2'deki en düşük `Selection RMSE` ile yapıldı.

</details>

### İsteğe bağlı MLflow kaydı

Son hücreyi çalıştırarak baseline, arama, bağımsız XGBoost karşılaştırması ve final test sonuçlarını yerel `mlflow.db` dosyasına yazabilirsiniz. Sonra defterin çalışma dizininde `uv run mlflow ui --backend-store-uri sqlite:///mlflow.db` komutunu çalıştırın. Aynı hücreyi yeniden çalıştırmak yeni deney kayıtları ekler.

In [27]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Concrete_Strength_Regression_HPO")
for row in model_results.to_dict("records"):
    with mlflow.start_run(run_name=row["Model"]):
        mlflow.log_metric("baseline_validation_rmse", row["Baseline RMSE"])
        if not pd.isna(row["Tuned RMSE"]):
            mlflow.log_metric("hpo_selection_rmse", row["Tuned RMSE"])
            mlflow.log_params({k: str(v) for k, v in row["Best parameters"].items()})

with mlflow.start_run(run_name="XGBoost – nested CV"):
    mlflow.log_metrics({
        "baseline_outer_rmse": xgb_summary.loc["Before HPO", "Validation RMSE"],
        "hpo_outer_rmse": xgb_summary.loc["After HPO", "Validation RMSE"],
    })

with mlflow.start_run(run_name="Final test"):
    mlflow.log_param("model", selected_name)
    mlflow.log_metrics({"test_rmse": test_rmse, "test_r2": test_r2})

c:\Users\Asus\Desktop\Personal\ml_crash_course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
